# Assignment 2

Perform linear regression with L1 (Lasso) and L2 (Ridge) regularization to predict the price of a House. Use hyper-parameter tuning for the best result. Evaluate the accuracy of the models by computing relevant performance metrics, including the R^2 value. Generate and display a plot that compares the actual values to the predicted values (Actual vs Predicted) for both tasks.
Dataset: housing_price_dataset.csv

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error
)

# 1. Load Real Dataset & One-Hot Encode Categorical Variables
df = pd.read_csv('housing_price_dataset.csv')
df_enc = pd.get_dummies(df, drop_first=True)
X, y = df_enc.drop(columns=['price']), df_enc['price']

# 2. Standardize Features and Train-Test Split (80/20)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# 3. Hyperparameter Tuning using GridSearchCV
param_grid = {'alpha': np.logspace(-2, 3, 30)}
ridge_cv = GridSearchCV(
    Ridge(), param_grid, cv=5, scoring='neg_mean_squared_error'
).fit(X_train, y_train)
lasso_cv = GridSearchCV(
    Lasso(max_iter=5000), param_grid, cv=5, scoring='neg_mean_squared_error'
).fit(X_train, y_train)

best_ridge = ridge_cv.best_estimator_
best_lasso = lasso_cv.best_estimator_

# 4. Evaluation on Real Test Set
y_pred_ridge = best_ridge.predict(X_test)
y_pred_lasso = best_lasso.predict(X_test)

def get_metrics(name, y_true, y_pred, model):
    return {'Model': name, 'Alpha': round(model.alpha, 3),
            'R^2': r2_score(y_true, y_pred),
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred))}

results = pd.DataFrame([
    get_metrics('Ridge (L2)', y_test, y_pred_ridge, best_ridge),
    get_metrics('Lasso (L1)', y_test, y_pred_lasso, best_lasso)
])
print("=== Real Data: Housing Price Regularization Metrics ===")
print(results.to_string(index=False))

# 5. Side-by-Side Actual vs Predicted Comparison Plot
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
models_data = [(axes[0], y_pred_ridge, 'Ridge (L2)', 'royalblue'),
               (axes[1], y_pred_lasso, 'Lasso (L1)', 'crimson')]
for ax, y_p, title, col in models_data:
    r2_val = r2_score(y_test, y_p)
    ax.scatter(y_test/1e6, y_p/1e6, color=col, alpha=0.6, edgecolors='k')
    p_min, p_max = y_test.min()/1e6, y_test.max()/1e6
    ax.plot([p_min, p_max], [p_min, p_max], 'k--', lw=2)
    ax.set_title(f"{title} [R^2 = {r2_val:.4f}]")
    ax.set_xlabel('Actual Price (M)'); ax.set_ylabel('Predicted Price (M)')
    ax.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout(); plt.savefig('ridge_lasso_comparison.png', dpi=300)
plt.show()